# Programming Paradigms as Cognitive Instruments

## Comparative utility, trade-offs, executable thought experiments, and proposed experimental paradigms

This notebook studies programming paradigms not merely as implementation styles, but as **ways of decomposing problems**.

The core question is:

> Which paradigm makes which kinds of structure easiest to see, express, verify, transform, or maintain?

The notebook compares twelve paradigms:

1. Procedural / imperative
2. Object-oriented
3. Functional
4. Logic programming
5. Declarative query programming
6. Constraint programming
7. Dataflow / reactive programming
8. Event-driven programming
9. Actor-model concurrency
10. CSP / channel-based concurrency
11. Array / vector programming
12. Concatenative / stack-based programming

The experiments are deliberately small. They are not intended to prove that one paradigm is universally superior. Instead, they expose differences in cognitive organization: explicit state, local reasoning, compositionality, inversion of control, search, concurrency structure, algebraic compactness, and the distance between a problem statement and an executable representation.

The final section proposes several **experimental paradigms** that combine useful properties of existing ones.

In [ ]:
from __future__ import annotations

import math
import time
import random
import statistics
from dataclasses import dataclass
from collections import defaultdict, deque
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = random.Random(20260824)
np_rng = np.random.default_rng(20260824)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

## 1. Paradigm catalogue

The table below separates **what a paradigm makes primary** from the kinds of problems it tends to compress well.

In [ ]:
paradigms = pd.DataFrame([
    {
        "paradigm": "Procedural / Imperative",
        "primary_unit": "sequence of state-changing commands",
        "strong_for": "stepwise algorithms, systems work, explicit control flow",
        "main_tradeoff": "state interactions grow difficult to reason about",
        "thinking_style": "simulation through time",
    },
    {
        "paradigm": "Object-Oriented",
        "primary_unit": "stateful entities with encapsulated behavior",
        "strong_for": "domain models, evolving entities, interface boundaries",
        "main_tradeoff": "hidden mutation and inheritance can obscure causality",
        "thinking_style": "world of interacting things",
    },
    {
        "paradigm": "Functional",
        "primary_unit": "composition of transformations",
        "strong_for": "data transformation, parallelism, local reasoning",
        "main_tradeoff": "effects and stateful workflows require additional structure",
        "thinking_style": "algebra of transformations",
    },
    {
        "paradigm": "Logic",
        "primary_unit": "facts, relations, and inference rules",
        "strong_for": "search, symbolic reasoning, relational structure",
        "main_tradeoff": "operational behavior may be less obvious",
        "thinking_style": "what must be true",
    },
    {
        "paradigm": "Declarative Query",
        "primary_unit": "desired relation or result set",
        "strong_for": "selection, aggregation, relational analytics",
        "main_tradeoff": "execution strategy is delegated to an engine",
        "thinking_style": "state the result, not the procedure",
    },
    {
        "paradigm": "Constraint",
        "primary_unit": "variables plus admissibility conditions",
        "strong_for": "scheduling, configuration, combinatorial search",
        "main_tradeoff": "solver behavior can feel opaque",
        "thinking_style": "define the admissible state space",
    },
    {
        "paradigm": "Dataflow / Reactive",
        "primary_unit": "dependency graph and propagation",
        "strong_for": "spreadsheets, UI state, streaming transformations",
        "main_tradeoff": "feedback and hidden dependencies can become difficult",
        "thinking_style": "change propagates through relations",
    },
    {
        "paradigm": "Event-Driven",
        "primary_unit": "handlers responding to events",
        "strong_for": "GUIs, servers, device interfaces, asynchronous systems",
        "main_tradeoff": "global behavior emerges from dispersed callbacks",
        "thinking_style": "react to occurrences",
    },
    {
        "paradigm": "Actor Model",
        "primary_unit": "isolated agents exchanging messages",
        "strong_for": "distributed systems, fault isolation, concurrency",
        "main_tradeoff": "ordering and eventual behavior can be hard to inspect",
        "thinking_style": "independent agents communicating",
    },
    {
        "paradigm": "CSP / Channels",
        "primary_unit": "processes synchronized by communication",
        "strong_for": "pipelines, concurrency protocols, structured coordination",
        "main_tradeoff": "protocol design can dominate local computation",
        "thinking_style": "processes coordinated through explicit channels",
    },
    {
        "paradigm": "Array / Vector",
        "primary_unit": "whole-array transformations",
        "strong_for": "numerical work, tensor operations, signal/image processing",
        "main_tradeoff": "indexing and shape semantics can become implicit",
        "thinking_style": "operate on structures all at once",
    },
    {
        "paradigm": "Concatenative / Stack",
        "primary_unit": "composition of stack transformations",
        "strong_for": "compact interpreters, pipelines, compositional operators",
        "main_tradeoff": "stack effects can become mentally expensive",
        "thinking_style": "compose transformations by adjacency",
    },
])

paradigms

## 2. Relative utility matrix

The ratings below are hypotheses, not universal truths. They are useful as a starting model that can later be replaced with empirical user-study data.

In [ ]:
tasks = [
    "Numerical transformation",
    "Symbolic inference",
    "Stateful simulation",
    "Distributed concurrency",
    "Interactive UI",
    "Data querying",
    "Constraint satisfaction",
    "Streaming pipeline",
    "Formal local reasoning",
    "Rapid exploratory coding",
    "Domain modeling",
    "Compact operator composition",
]

ratings = {
    "Procedural / Imperative":       [4,2,5,2,3,2,2,3,2,5,3,2],
    "Object-Oriented":               [2,2,5,3,4,2,2,2,2,3,5,2],
    "Functional":                    [5,3,3,4,3,3,3,5,5,4,3,4],
    "Logic":                         [1,5,2,1,1,3,5,1,4,2,3,2],
    "Declarative Query":             [2,3,1,1,1,5,3,3,4,4,2,2],
    "Constraint":                    [1,4,2,2,1,2,5,1,4,2,3,1],
    "Dataflow / Reactive":           [3,2,3,3,5,3,2,5,3,4,2,3],
    "Event-Driven":                  [2,1,3,4,5,1,1,4,2,3,3,2],
    "Actor Model":                   [2,2,3,5,3,1,2,4,3,2,3,2],
    "CSP / Channels":                [3,2,3,5,2,1,2,5,4,2,2,3],
    "Array / Vector":                [5,1,2,2,1,3,1,4,4,5,1,5],
    "Concatenative / Stack":         [3,2,2,2,2,2,2,4,4,3,1,5],
}

utility_df = pd.DataFrame(ratings, index=tasks).T
utility_df

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(utility_df.values, aspect="auto")
plt.xticks(range(len(tasks)), tasks, rotation=75, ha="right")
plt.yticks(range(len(utility_df.index)), utility_df.index)
plt.colorbar(label="Hypothesized utility (1–5)")
plt.title("Paradigm Utility Matrix")
plt.tight_layout()
plt.show()

# Experiment 1 — State mutation versus transformation

The same task is written in imperative and functional form. The experiment counts how many intermediate states must be mentally tracked.

This is a crude metric, but it illustrates a meaningful distinction: imperative code often exposes *time*, while functional code exposes *composition*.

In [ ]:
data = [3, 7, 2, 9, 4, 11, 6]

def imperative_pipeline(xs):
    temp = []
    for x in xs:
        if x % 2 == 1:
            temp.append(x)
    for i in range(len(temp)):
        temp[i] = temp[i] * temp[i]
    total = 0
    for x in temp:
        total += x
    return total

def functional_pipeline(xs):
    return sum(x * x for x in xs if x % 2 == 1)

assert imperative_pipeline(data) == functional_pipeline(data)

experiment_01 = pd.DataFrame([
    {
        "style": "imperative",
        "named_mutable_states": 2,
        "explicit_loops": 3,
        "pipeline_visible_as_one_expression": False,
    },
    {
        "style": "functional",
        "named_mutable_states": 0,
        "explicit_loops": 0,
        "pipeline_visible_as_one_expression": True,
    },
])

experiment_01

# Experiment 2 — Object model versus relational representation

Object-oriented organization compresses reasoning when identity and state belong naturally to persistent entities. Relational organization can be clearer when the interesting structure lies in relations rather than objects.

In [ ]:
@dataclass
class Account:
    name: str
    balance: float

    def transfer_to(self, other, amount):
        self.balance -= amount
        other.balance += amount

alice = Account("Alice", 100)
bob = Account("Bob", 50)
alice.transfer_to(bob, 25)

object_state = {"Alice": alice.balance, "Bob": bob.balance}

accounts = {
    "Alice": {"balance": 100},
    "Bob": {"balance": 50},
}
transfers = [("Alice", "Bob", 25)]

for src, dst, amount in transfers:
    accounts[src]["balance"] -= amount
    accounts[dst]["balance"] += amount

relational_state = {k: v["balance"] for k, v in accounts.items()}

assert object_state == relational_state
object_state

# Experiment 3 — Forward procedure versus logic-style relation

Procedural code answers “how do I compute the output?” Logic programming instead asks “what assignments satisfy the relation?”

Here we simulate the difference using brute-force relational search in Python.

In [ ]:
def procedural_square_root_of_49():
    return math.sqrt(49)

def relational_square_relation(limit=20):
    solutions = []
    for x in range(-limit, limit + 1):
        if x * x == 49:
            solutions.append(x)
    return solutions

pd.DataFrame([
    {
        "form": "procedural",
        "question": "compute sqrt(49)",
        "answers": [procedural_square_root_of_49()],
        "directionality": "forward",
    },
    {
        "form": "relational",
        "question": "find x where x*x = 49",
        "answers": relational_square_relation(),
        "directionality": "bidirectional relation",
    },
])

# Experiment 4 — Constraint thinking versus algorithm design

A scheduling problem can be attacked by inventing a search procedure or by declaring admissibility conditions.

This experiment encodes the latter and counts how much of the candidate state space survives each added constraint.

In [ ]:
people = ["A", "B", "C"]
slots = [1, 2, 3]

assignments = [
    dict(zip(people, perm))
    for perm in __import__("itertools").permutations(slots)
]

stages = []

current = assignments
stages.append(("all assignments", len(current)))

current = [a for a in current if a["A"] != 1]
stages.append(("A not in slot 1", len(current)))

current = [a for a in current if a["B"] < a["C"]]
stages.append(("B before C", len(current)))

current = [a for a in current if a["A"] != a["C"]]
stages.append(("A differs from C", len(current)))

constraint_df = pd.DataFrame(stages, columns=["constraint_stage", "remaining_states"])
constraint_df

# Experiment 5 — Query thinking versus loop thinking

Declarative query languages compress “what subset and aggregation do I want?” into relational operators. Imperative code makes iteration explicit.

Pandas is used here as a query-like stand-in.

In [ ]:
sales = pd.DataFrame({
    "region": ["N", "S", "N", "E", "S", "E"],
    "amount": [10, 20, 15, 5, 25, 30],
})

imperative_totals = defaultdict(int)
for row in sales.to_dict("records"):
    if row["amount"] >= 15:
        imperative_totals[row["region"]] += row["amount"]

query_totals = (
    sales[sales["amount"] >= 15]
    .groupby("region")["amount"]
    .sum()
    .to_dict()
)

assert dict(imperative_totals) == query_totals
query_totals

# Experiment 6 — Dataflow and change propagation

In dataflow thinking, values are understood through dependencies. This experiment contrasts recomputing an entire pipeline with propagating only affected nodes.

In [ ]:
deps = {
    "a": [],
    "b": [],
    "sum": ["a", "b"],
    "double": ["sum"],
    "offset": ["double"],
}

def evaluate_graph(a, b):
    values = {"a": a, "b": b}
    values["sum"] = values["a"] + values["b"]
    values["double"] = 2 * values["sum"]
    values["offset"] = values["double"] + 3
    return values

before = evaluate_graph(2, 5)
after = evaluate_graph(2, 8)

changed_nodes = [k for k in before if before[k] != after[k]]

pd.DataFrame({
    "node": list(before.keys()),
    "before": [before[k] for k in before],
    "after": [after[k] for k in after],
    "changed": [k in changed_nodes for k in before],
})

# Experiment 7 — Event-driven inversion of control

Event-driven systems organize computation around external occurrences rather than a central sequential procedure.

The experiment shows how a single event log can trigger different handlers without the event producer knowing their implementation.

In [ ]:
handlers = defaultdict(list)
event_log = []

def on(event_name, fn):
    handlers[event_name].append(fn)

def emit(event_name, payload):
    event_log.append((event_name, payload))
    for fn in handlers[event_name]:
        fn(payload)

state = {"count": 0, "audit": []}

on("click", lambda payload: state.__setitem__("count", state["count"] + 1))
on("click", lambda payload: state["audit"].append(("click", payload)))

emit("click", {"x": 10, "y": 20})
emit("click", {"x": 11, "y": 21})

state

# Experiment 8 — Actor isolation versus shared mutable state

This simplified actor simulation replaces shared-state mutation with message passing.

The question is not speed. It is whether the ownership boundary makes causality easier to localize.

In [ ]:
class CounterActor:
    def __init__(self):
        self.mailbox = deque()
        self.value = 0

    def send(self, message):
        self.mailbox.append(message)

    def step(self):
        if not self.mailbox:
            return
        msg = self.mailbox.popleft()
        if msg == "inc":
            self.value += 1
        elif isinstance(msg, tuple) and msg[0] == "add":
            self.value += msg[1]

actor = CounterActor()
for _ in range(5):
    actor.send("inc")
actor.send(("add", 10))

while actor.mailbox:
    actor.step()

actor.value

# Experiment 9 — CSP / channels and explicit coordination

Channel-based concurrency makes communication structure explicit. The pipeline below models producer → transformer → consumer as distinct processes connected by queues.

In [ ]:
producer_to_worker = deque()
worker_to_consumer = deque()

def producer(xs):
    for x in xs:
        producer_to_worker.append(x)

def worker():
    while producer_to_worker:
        x = producer_to_worker.popleft()
        worker_to_consumer.append(x * x)

def consumer():
    out = []
    while worker_to_consumer:
        out.append(worker_to_consumer.popleft())
    return out

producer([1, 2, 3, 4])
worker()
channel_result = consumer()
channel_result

# Experiment 10 — Array thinking versus scalar iteration

Array programming turns repeated scalar operations into transformations over entire structures.

This experiment compares the representations and times them on a moderately large numerical task.

In [ ]:
n = 500_000
xs_list = list(range(n))
xs_array = np.arange(n)

t0 = time.perf_counter()
scalar_result = [x * x + 2 * x + 1 for x in xs_list]
scalar_time = time.perf_counter() - t0

t0 = time.perf_counter()
array_result = xs_array * xs_array + 2 * xs_array + 1
array_time = time.perf_counter() - t0

assert scalar_result[-1] == int(array_result[-1])

pd.DataFrame([{
    "scalar_seconds": scalar_time,
    "array_seconds": array_time,
    "speedup": scalar_time / array_time if array_time else float("inf"),
}])

# Experiment 11 — Concatenative composition

Concatenative programming treats programs as compositions of stack transformations. The absence of named intermediate variables can make operator structure unusually visible, while increasing the burden of tracking stack effects.

In [ ]:
def run_stack_program(initial_stack, program):
    stack = list(initial_stack)

    for op in program:
        if callable(op):
            op(stack)
        else:
            stack.append(op)

    return stack

def add(stack):
    b = stack.pop()
    a = stack.pop()
    stack.append(a + b)

def mul(stack):
    b = stack.pop()
    a = stack.pop()
    stack.append(a * b)

# Equivalent to: (3 + 4) * 5
stack_result = run_stack_program([], [3, 4, add, 5, mul])

stack_result

# Experiment 12 — Which paradigm exposes errors earliest?

Different paradigms make different classes of mistakes visible at different stages.

This experiment uses a synthetic scoring model for common failure classes. The values are hypotheses that can later be replaced by empirical study.

In [ ]:
failure_classes = [
    "state mutation bug",
    "wrong relation",
    "invalid configuration",
    "race condition",
    "shape mismatch",
    "event ordering bug",
    "hidden dependency",
]

detection_scores = pd.DataFrame({
    "Procedural":       [2,2,2,1,2,2,2],
    "Object-Oriented":  [3,2,2,2,2,2,2],
    "Functional":       [5,3,3,4,3,3,4],
    "Logic":            [2,5,4,1,1,1,3],
    "Constraint":       [2,4,5,2,1,1,4],
    "Dataflow":         [3,2,3,3,2,4,5],
    "Actor":            [4,2,3,5,1,3,3],
    "CSP":              [4,2,3,5,1,4,3],
    "Array":            [3,1,2,2,5,1,2],
    "Concatenative":    [3,2,2,2,3,2,3],
}, index=failure_classes)

detection_scores

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(detection_scores.values, aspect="auto")
plt.xticks(range(len(detection_scores.columns)), detection_scores.columns, rotation=60, ha="right")
plt.yticks(range(len(detection_scores.index)), detection_scores.index)
plt.colorbar(label="Hypothesized early-detection utility")
plt.title("Experiment 12: Paradigms and error visibility")
plt.tight_layout()
plt.show()

# 13. Cognitive dimensions

A paradigm can be analyzed independently of task performance.

The following dimensions describe the kind of mental compression each paradigm encourages:

**Temporal explicitness** — how directly the code represents sequence through time.

**State locality** — how easy it is to identify who owns mutable state.

**Relational expressiveness** — how naturally the language describes relations rather than procedures.

**Compositionality** — how easily large programs are built from small independent parts.

**Search inversion** — whether the programmer specifies conditions and lets the system find solutions.

**Concurrency legibility** — whether coordination is explicit in the program structure.

**Shape-level thinking** — whether the programmer operates on whole structures rather than scalar elements.

**Effect visibility** — whether mutation, I/O, and other effects remain explicit.

In [ ]:
cognitive_dimensions = pd.DataFrame({
    "Paradigm": [
        "Procedural", "Object-Oriented", "Functional", "Logic",
        "Declarative Query", "Constraint", "Dataflow", "Event-Driven",
        "Actor", "CSP", "Array", "Concatenative"
    ],
    "Temporal explicitness": [5,4,2,1,1,1,3,4,4,4,1,2],
    "State locality": [2,4,5,4,5,4,3,2,5,5,4,4],
    "Relational expressiveness": [2,3,3,5,5,5,4,2,2,3,3,2],
    "Compositionality": [3,3,5,4,4,4,5,3,4,5,5,5],
    "Search inversion": [1,1,2,5,4,5,2,1,1,1,1,1],
    "Concurrency legibility": [2,3,4,2,2,3,4,3,5,5,4,3],
    "Shape-level thinking": [2,2,3,1,3,2,4,2,2,3,5,4],
    "Effect visibility": [3,2,5,4,4,4,3,2,4,5,4,4],
}).set_index("Paradigm")

cognitive_dimensions

# 14. Task-to-paradigm recommendation experiment

Given a task profile, we can score paradigms by matching the task's cognitive demands against the paradigm's hypothesized strengths.

This is not a recommendation engine in any authoritative sense. It is a way to make the comparison explicit and inspectable.

In [ ]:
def recommend_paradigms(task_weights):
    rows = []
    for paradigm, row in cognitive_dimensions.iterrows():
        score = 0.0
        for dimension, weight in task_weights.items():
            score += row[dimension] * weight
        rows.append((paradigm, score))
    return pd.DataFrame(rows, columns=["paradigm", "score"]).sort_values("score", ascending=False)

example_task = {
    "State locality": 2,
    "Compositionality": 3,
    "Concurrency legibility": 5,
    "Effect visibility": 4,
}

recommend_paradigms(example_task)

# 15. Proposed empirical user-study paradigms

The notebook so far uses code-level proxies. A serious study of “which paradigm is good for which type of thinking” should include human experiments.

A useful protocol would give programmers equivalent problems in different paradigms and measure not only completion time, but **error localization, revision cost, explanation quality, transfer to a related problem, confidence calibration, and retention after delay**.

Several especially useful experimental designs follow.

In [ ]:
proposed_user_studies = pd.DataFrame([
    {
        "study": "State Reconstruction",
        "question": "Which paradigm helps users reconstruct program state after interruption?",
        "comparison": "imperative vs functional vs actor",
        "measure": "state-reconstruction accuracy after a timed distraction",
    },
    {
        "study": "Relation Inversion",
        "question": "Which paradigm helps users reason backward from desired output to admissible inputs?",
        "comparison": "procedural vs logic vs constraint",
        "measure": "solution accuracy and number of reformulations",
    },
    {
        "study": "Change Propagation",
        "question": "Which paradigm best supports reasoning about consequences of one changed input?",
        "comparison": "imperative vs reactive/dataflow vs spreadsheet-like",
        "measure": "correct prediction of downstream changes",
    },
    {
        "study": "Concurrency Explanation",
        "question": "Which paradigm makes concurrent behavior easiest to explain?",
        "comparison": "shared-state threads vs actors vs CSP",
        "measure": "causal explanation quality and race-condition detection",
    },
    {
        "study": "Representation Transfer",
        "question": "Does learning one paradigm improve thinking in another domain?",
        "comparison": "array vs scalar; logic vs procedural; functional vs imperative",
        "measure": "performance on structurally related non-programming tasks",
    },
    {
        "study": "Compression Versus Recoverability",
        "question": "When does concise code become too compressed to reconstruct mentally?",
        "comparison": "array, concatenative, functional point-free, explicit procedural",
        "measure": "time to explain code correctly after delay",
    },
])

proposed_user_studies

# 16. Proposed experimental programming paradigms

The existing paradigm taxonomy is not exhaustive. Several hybrids are worth testing precisely because they reorganize **what the programmer is asked to specify**.

In [ ]:
experimental_paradigms = pd.DataFrame([
    {
        "name": "Admissibility Programming",
        "core_idea": "Programs specify allowed state transitions rather than commands.",
        "inherits_from": "constraint + state machines + logic",
        "experimental_question": "Can complex systems become easier to verify when transition legality is primary?",
    },
    {
        "name": "Evidence-Carrying Programming",
        "core_idea": "Every derived value carries provenance and transformation history.",
        "inherits_from": "functional + dataflow + proof-carrying systems",
        "experimental_question": "Does explicit provenance improve debugging and trust calibration?",
    },
    {
        "name": "Repair-Oriented Programming",
        "core_idea": "Components specify invariants and local repair strategies rather than only normal execution.",
        "inherits_from": "fault tolerance + actors + constraint systems",
        "experimental_question": "Can repair semantics reduce global exception-handling complexity?",
    },
    {
        "name": "Bidirectional Transformation Programming",
        "core_idea": "Relations are defined so updates can propagate in more than one direction.",
        "inherits_from": "lenses + logic + reactive systems",
        "experimental_question": "Can synchronization tasks be expressed without privileging one direction of computation?",
    },
    {
        "name": "History-Sensitive Programming",
        "core_idea": "The admissibility of an operation depends explicitly on the path by which state was reached.",
        "inherits_from": "event sourcing + typestate + temporal logic",
        "experimental_question": "Does explicit historical state make protocol and security logic clearer?",
    },
    {
        "name": "Operator-First Programming",
        "core_idea": "Programs are assembled from named transformations with declared algebraic properties.",
        "inherits_from": "functional + concatenative + array programming",
        "experimental_question": "Can declared operator laws improve composition and automated optimization?",
    },
    {
        "name": "Distinction-Oriented Programming",
        "core_idea": "Programs manipulate distinctions, equivalence classes, and refinements directly.",
        "inherits_from": "type systems + logic + abstract interpretation",
        "experimental_question": "Is refinement of distinctions a better primitive for classification and analysis tasks?",
    },
    {
        "name": "Boundary-Oriented Programming",
        "core_idea": "Capabilities, admissibility, and interface boundaries are first-class program elements.",
        "inherits_from": "capability systems + actors + effect systems",
        "experimental_question": "Can system behavior be understood better when boundaries are explicit rather than ambient?",
    },
])

experimental_paradigms

# 17. Experimental paradigm prototype: admissible transitions

This toy interpreter treats the program as a set of admissible transitions. Instead of directly commanding the machine, we ask which next states are legal.

In [ ]:
@dataclass(frozen=True)
class State:
    value: int
    mode: str

def admissible_transitions(state):
    next_states = []

    if state.mode == "normal":
        next_states.append(State(state.value + 1, "normal"))

    if state.value >= 3 and state.mode == "normal":
        next_states.append(State(state.value, "locked"))

    if state.mode == "locked" and state.value == 3:
        next_states.append(State(state.value, "normal"))

    return next_states

trajectory = [State(0, "normal")]

for _ in range(5):
    options = admissible_transitions(trajectory[-1])
    if not options:
        break
    trajectory.append(options[0])

trajectory

# 18. Experimental paradigm prototype: evidence-carrying values

The value below retains a trace of how it was produced. This is not sophisticated provenance, but it shows what changes when history is treated as part of the computational object.

In [ ]:
@dataclass
class EvidenceValue:
    value: Any
    history: list[str]

def ev(value, label):
    return EvidenceValue(value, [label])

def ev_map(x, fn, label):
    return EvidenceValue(fn(x.value), x.history + [label])

x = ev(5, "input:x=5")
y = ev_map(x, lambda n: n * 2, "double")
z = ev_map(y, lambda n: n + 3, "add 3")

z

# 19. Experimental paradigm prototype: repair-oriented state

A repair-oriented object does not only say what valid state is. It also specifies a local operation for returning to admissibility.

In [ ]:
@dataclass
class RepairableCounter:
    value: int
    lower: int = 0
    upper: int = 10

    def valid(self):
        return self.lower <= self.value <= self.upper

    def repair(self):
        self.value = min(max(self.value, self.lower), self.upper)
        return self

counter = RepairableCounter(17)
before = counter.value
counter.repair()
after = counter.value

{"before": before, "after": after, "valid_after_repair": counter.valid()}

# 20. Synthesis

The twelve paradigms can be understood as choosing different **primary objects of thought**:

Procedural programming privileges **actions**.

Object-oriented programming privileges **entities**.

Functional programming privileges **transformations**.

Logic programming privileges **relations**.

Declarative query programming privileges **desired result sets**.

Constraint programming privileges **admissible states**.

Dataflow programming privileges **dependencies**.

Event-driven programming privileges **occurrences**.

Actor programming privileges **isolated agents**.

CSP privileges **communication protocols**.

Array programming privileges **whole structures**.

Concatenative programming privileges **operator composition**.

This suggests that the question “Which programming paradigm is best?” is badly formed. A more productive question is:

> Which ontology of computation most closely matches the structure of the problem currently being reasoned about?

The experimental paradigms proposed here push that question further by making **admissibility, evidence, repair, history, distinction, and boundaries** candidates for first-class computational primitives.

# 21. Export core tables

In [ ]:
OUTPUT_DIR = "programming_paradigm_experiments"
os.makedirs(OUTPUT_DIR, exist_ok=True)

exports = {
    "paradigm_catalogue.csv": paradigms,
    "utility_matrix.csv": utility_df.reset_index(names="paradigm"),
    "cognitive_dimensions.csv": cognitive_dimensions.reset_index(),
    "constraint_experiment.csv": constraint_df,
    "experimental_paradigms.csv": experimental_paradigms,
    "proposed_user_studies.csv": proposed_user_studies,
}

for filename, df in exports.items():
    df.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

print(f"Exported {len(exports)} tables to {OUTPUT_DIR}/")